<a href="https://colab.research.google.com/github/oesquivel81/TopoRAG/blob/main/01_document_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
    arxiv \
    pymupdf \
    pandas \
    pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 66.6 MB/s eta 0:00:00


In [14]:
from pathlib import Path

import arxiv
import fitz
import pandas as pd
import requests

In [3]:
search = arxiv.Search(
    query='cat:math.DG AND all:"differential geometry"',
    max_results=3,
    sort_by=arxiv.SortCriterion.Relevance,
)

In [4]:
client = arxiv.Client(
    page_size=10,
    delay_seconds=3,
    num_retries=3,
)

In [5]:
results = list(client.results(search))

print(f"Articles found: {len(results)}")

Articles found: 3


In [6]:
for index, paper in enumerate(results, start=1):
    print("=" * 100)
    print(f"Paper {index}")
    print(f"Title: {paper.title}")
    print(f"Authors: {', '.join(author.name for author in paper.authors)}")
    print(f"Published: {paper.published}")
    print(f"arXiv ID: {paper.entry_id}")
    print(f"Primary category: {paper.primary_category}")
    print()
    print(paper.summary[:800])

Paper 1
Title: Discrete differential geometry. Consistency as integrability
Authors: Alexander I. Bobenko, Yuri B. Suris
Published: 2005-04-18 11:16:37+00:00
arXiv ID: http://arxiv.org/abs/math/0504358v1
Primary category: math.DG

  A new field of discrete differential geometry is presently emerging on the border between differential and discrete geometry. Whereas classical differential geometry investigates smooth geometric shapes (such as surfaces), and discrete geometry studies geometric shapes with finite number of elements (such as polyhedra), the discrete differential geometry aims at the development of discrete equivalents of notions and methods of smooth surface theory. Current interest in this field derives not only from its importance in pure mathematics but also from its relevance for other fields like computer graphics. Recent progress in discrete differential geometry has lead, somewhat unexpectedly, to a better understanding of some fundamental structures lying in the bas

In [7]:
records = []

for paper in results:
    records.append({
        "title": paper.title,
        "authors": [
            author.name
            for author in paper.authors
        ],
        "published": paper.published,
        "updated": paper.updated,
        "entry_id": paper.entry_id,
        "pdf_url": paper.pdf_url,
        "primary_category": paper.primary_category,
        "categories": paper.categories,
        "summary": paper.summary,
        "topic": "differential_geometry",
        "source_type": "arxiv",
    })

metadata_df = pd.DataFrame(records)

metadata_df[
    [
        "title",
        "published",
        "primary_category",
        "pdf_url",
    ]
]

,title,published,primary_category,pdf_url
0,Discrete differential geometry. Consistency as...,2005-04-18 11:16:37+00:00,math.DG,https://arxiv.org/pdf/math/0504358v1
1,"A translation of ""What is differential geometr...",2026-01-05 18:16:44+00:00,math.HO,https://arxiv.org/pdf/2601.02325v1
2,Bonnet's type theorems in the relative differe...,2017-07-21 06:00:33+00:00,math.DG,https://arxiv.org/pdf/1707.07549v4


In [9]:
PROJECT_ROOT = Path("/content/TopoRAG")
PDF_DIR = PROJECT_ROOT / "data" / "raw" / "arxiv"

PDF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [10]:
def extract_arxiv_id(entry_id: str) -> str:
    return entry_id.rstrip("/").split("/")[-1]

In [15]:
downloaded_files = []

for paper in results:
    arxiv_id = extract_arxiv_id(paper.entry_id)

    output_file = PDF_DIR / f"{arxiv_id}.pdf"

    # Use requests to download the PDF
    response = requests.get(paper.pdf_url)
    response.raise_for_status() # Raise an exception for bad status codes

    with open(output_file, 'wb') as f:
        f.write(response.content)

    downloaded_files.append(output_file)

    print(f"Downloaded: {output_file.name}")

Downloaded: 0504358v1.pdf
Downloaded: 2601.02325v1.pdf
Downloaded: 1707.07549v4.pdf


In [16]:
def extract_pdf_text(pdf_path: Path) -> list[dict]:
    pages = []

    with fitz.open(pdf_path) as pdf:
        total_pages = len(pdf)

        for page_index, page in enumerate(pdf):
            text = page.get_text("text")

            pages.append({
                "source_file": pdf_path.name,
                "page_number": page_index + 1,
                "total_pages": total_pages,
                "page_content": text.strip(),
            })

    return pages

In [17]:
page_records = []

for pdf_path in downloaded_files:
    pages = extract_pdf_text(pdf_path)
    page_records.extend(pages)

print(f"Pages extracted: {len(page_records)}")

Pages extracted: 384


In [21]:
metadata_df["arxiv_id"] = (
    metadata_df["entry_id"]
    .apply(extract_arxiv_id)
)

pages_df = pd.DataFrame(page_records)

pages_df["arxiv_id"] = (
    pages_df["source_file"]
    .str.replace(".pdf", "", regex=False)
)

pages_df["word_count"] = pages_df["page_content"].apply(lambda x: len(x.split()))

In [22]:
corpus_df = pages_df.merge(
    metadata_df,
    on="arxiv_id",
    how="left",
)

corpus_df[
    [
        "title",
        "page_number",
        "word_count",
        "primary_category",
    ]
].head(10)

,title,page_number,word_count,primary_category
0,Discrete differential geometry. Consistency as...,1,36,math.DG
1,Discrete differential geometry. Consistency as...,2,1,math.DG
2,Discrete differential geometry. Consistency as...,3,583,math.DG
3,Discrete differential geometry. Consistency as...,4,373,math.DG
4,Discrete differential geometry. Consistency as...,5,282,math.DG
5,Discrete differential geometry. Consistency as...,6,68,math.DG
6,Discrete differential geometry. Consistency as...,7,417,math.DG
7,Discrete differential geometry. Consistency as...,8,291,math.DG
8,Discrete differential geometry. Consistency as...,9,416,math.DG
9,Discrete differential geometry. Consistency as...,10,426,math.DG


In [23]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [24]:
output_path = (
    PROCESSED_DIR
    / "arxiv_differential_geometry_pages.parquet"
)

corpus_df.to_parquet(
    output_path,
    index=False,
)

print(f"Saved: {output_path}")

Saved: /content/TopoRAG/data/processed/arxiv_differential_geometry_pages.parquet
